# F1 2026 Constructors Title Odds
## Simulating the Remaining Races from the Weekly Tracker

# Data Prep
## Imports

In [1]:
import pandas as pd
import numpy as np

## Load Race History

In [2]:
results = pd.read_excel('F1_2026_Constructors_Data.xlsx', sheet_name='Race_Results_1_17', header=[0,1])

teams = ['Mercedes', 'Ferrari', 'McLaren', 'Red Bull']

team_points = {}
for team in teams:
    team_points[team] = results[(team, 'Race_Points')].values

print(results.shape)
print({t: len(team_points[t]) for t in teams})

(17, 15)
{'Mercedes': 17, 'Ferrari': 17, 'McLaren': 17, 'Red Bull': 17}


In [3]:
raw = pd.read_excel('F1_2026_Constructors_Tracker.xlsx', sheet_name='Weekly_Points_2026', header=None)

print(raw.shape)
raw.iloc[:, :7]

(9, 15)


,0,1,2,3,4,5,6
0,Race_Number,Race_Name,Sprint_Weekend,Mercedes,NaN,NaN,Ferrari
1,NaN,NaN,NaN,Points_Scored,Cumulative_Points,Gap_to_Leader,Points_Scored
2,Start,Standings thru Race 17,NaN,578,578,NaN,444
3,18,United States GP,No,29,607,0,30
4,19,Gran Premio De Mexico,No,23,630,0,30
5,20,Sao Paula GP,No,22,652,0,6
6,21,Las Vegas GP,No,14,666,0,7
7,22,Qatar GP,No,NaN,NaN,NaN,NaN
8,23,Abu Dhabi GP,No,NaN,NaN,NaN,NaN


## Load Tracker

In [4]:
baseline = {}
tracker_points = {}

for k, team in enumerate(teams):
    baseline[team] = float(raw.iloc[2, 4 + 3*k])
    tracker_points[team] = pd.to_numeric(raw.iloc[3:, 3 + 3*k], errors='coerce')

print(baseline)
print({t: tracker_points[t].notna().sum() for t in teams})

{'Mercedes': 578.0, 'Ferrari': 444.0, 'McLaren': 339.0, 'Red Bull': 332.0}
{'Mercedes': np.int64(4), 'Ferrari': np.int64(4), 'McLaren': np.int64(4), 'Red Bull': np.int64(4)}


## Combine With Tracker

In [5]:
current_points = {}
for team in teams:
    current_points[team] = baseline[team] + tracker_points[team].sum()

total_races_in_tracker = len(raw.iloc[3:])
races_entered = tracker_points['Mercedes'].notna().sum()
n_remaining = int(total_races_in_tracker - races_entered)

print(current_points)
print("Races entered:", races_entered)
print("Races remaining:", n_remaining)

{'Mercedes': np.float64(666.0), 'Ferrari': np.float64(517.0), 'McLaren': np.float64(431.0), 'Red Bull': np.float64(440.0)}
Races entered: 4
Races remaining: 2


## Check Against Official Standings

In [11]:
title_race = pd.read_excel('F1_2026_Constructors_Tracker.xlsx', sheet_name='Title_Race')

print(title_race.shape)
title_race

(11, 6)


,Team,Points_Latest,Gap_to_Leader,Status,Official_Latest,Check_vs_Sheet
0,Mercedes,666,0.0,Leader,666.0,OK
1,Ferrari,517,-149.0,Eliminated,517.0,OK
2,McLaren,431,-235.0,Eliminated,431.0,OK
3,Red Bull,440,-226.0,Eliminated,440.0,OK
4,NaN,NaN,NaN,NaN,NaN,NaN
5,Races left,2,NaN,NaN,NaN,NaN
6,Races played (of 23),21,NaN,NaN,NaN,NaN
7,Lead over 2nd place,149,NaN,NaN,NaN,NaN
8,Max points still available,86,NaN,NaN,NaN,NaN
9,Title mathematically clinched?,YES,NaN,NaN,NaN,NaN


In [12]:
official = title_race.iloc[:4].set_index('Team')['Official_Latest']

check = pd.DataFrame({'Notebook': pd.Series(current_points), 'Official': official})
check['Match'] = check['Notebook'] == check['Official']

print(check)

          Notebook  Official  Match
Mercedes     666.0     666.0   True
Ferrari      517.0     517.0   True
McLaren      431.0     431.0   True
Red Bull     440.0     440.0   True


## Current Standings

In [13]:
standings = pd.DataFrame({'Points': pd.Series(current_points)}).sort_values('Points', ascending=False)
standings['Gap_to_Leader'] = standings['Points'] - standings['Points'].iloc[0]

print(standings)

          Points  Gap_to_Leader
Mercedes   666.0            0.0
Ferrari    517.0         -149.0
Red Bull   440.0         -226.0
McLaren    431.0         -235.0


# Monte Carlo Simulation
## Full Simulation

In [ ]:
# Seeded random generator, so the simulation is reproducible:
# re-running the notebook gives the same draws and the same results.
rng = np.random.default_rng(2026)

num_simulations = 1_000_000

final_totals_matrix = np.zeros((num_simulations, len(teams)))

for i, team in enumerate(teams):
    all_draws = rng.choice(team_points[team], size=(num_simulations, n_remaining))
    simulated_remaining = all_draws.sum(axis=1)
    final_totals_matrix[:, i] = current_points[team] + simulated_remaining

print(final_totals_matrix.shape)
print(final_totals_matrix[:3])


# Findings
## Title Probability

In [ ]:
winner_indices = np.argmax(final_totals_matrix, axis=1)

win_counts = {team: (winner_indices == i).sum() for i, team in enumerate(teams)}
win_probabilities = {team: win_counts[team] / num_simulations for team in teams}

for team in teams:
    print(f"{team}: {win_counts[team]:,} wins ({win_probabilities[team]:.4%})")

## Note: the constructors title is already clinched

Mercedes lead Ferrari by 149 points with only 86 points still available in the last 2 races, so Mercedes win the title in 100% of simulations. The simulation above is therefore not really about *whether* Mercedes win — it is about the expected final margin and the runner-up race below.

## Expected Margin of Victory

In [ ]:
sorted_totals = np.sort(final_totals_matrix, axis=1)[:, ::-1]
margins = sorted_totals[:, 0] - sorted_totals[:, 1]

print(f"Average margin: {margins.mean():.1f}")
print(f"Smallest margin: {margins.min():.0f}")
print(f"Largest margin: {margins.max():.0f}")
print(f"5th percentile: {np.percentile(margins, 5):.0f}")
print(f"95th percentile: {np.percentile(margins, 95):.0f}")

## Points Needed to Mathematically Clinch the Title

In [ ]:
race_rows = raw.iloc[3:]
remaining_rows = race_rows[race_rows[3].isna()]

max_per_race = remaining_rows[2].apply(lambda x: 58 if x == 'Yes' else 43).values
max_available = max_per_race.sum()

leader = standings.index[0]
second = standings.index[1]
lead = standings['Points'].iloc[0] - standings['Points'].iloc[1]

print("Leader:", leader, "| Second:", second)
print("Races left:", len(max_per_race))
print("Max points still available:", max_available)
print("Lead over second:", lead)
print("Clinched already?", lead > max_available)
print("Extra lead still needed:", max_available - lead + 1)

## Earliest Possible Clinch Race

In [ ]:
n_sims_clinch = 100_000

remaining_max_after = np.array([max_per_race[k+1:].sum() for k in range(n_remaining)])

cumulative = {}
for team in teams:
    draws = rng.choice(team_points[team], size=(n_sims_clinch, n_remaining))
    cumulative[team] = current_points[team] + np.cumsum(draws, axis=1)

rivals = [t for t in teams if t != leader]
best_rival = np.maximum.reduce([cumulative[r] for r in rivals])

clinched_matrix = cumulative[leader] > best_rival + remaining_max_after

print(remaining_max_after)
print(clinched_matrix.shape)

In [ ]:
race_labels = [f"{name} ({int(num)})" for num, name in zip(remaining_rows[0], remaining_rows[1])]

first_clinch = clinched_matrix.argmax(axis=1)

for k, label in enumerate(race_labels):
    first_here = (first_clinch == k).mean()
    cumulative_here = clinched_matrix[:, k].mean()
    print(f"{label}: first clinch {first_here:.2%}   clinched by here {cumulative_here:.2%}")

# Runner-Up Race (P2)
## P2 Probabilities

In [ ]:
p2_teams = [t for t in standings.index if t != leader]
final_p2 = pd.DataFrame(final_totals_matrix, columns=teams)[p2_teams]

runner_up = final_p2.idxmax(axis=1)
p2_prob = runner_up.value_counts(normalize=True).reindex(p2_teams).fillna(0)

(p2_prob * 100).round(2)

## Expected Final Points and Margins

In [ ]:
print(final_p2.quantile([0.05, 0.5, 0.95]).round(0))

sorted_totals = np.sort(final_p2.values, axis=1)[:, ::-1]
margin = sorted_totals[:, 0] - sorted_totals[:, 1]

print(f"Average gap between P2 and the next-best contender: {margin.mean():.1f}")
print(f"Chance that gap is 10 points or less: {(margin <= 10).mean():.1%}")

## Points Needed to Mathematically Clinch P2

In [ ]:
max_available = max_per_race.sum()
pts = standings.loc[p2_teams, 'Points']

lead_matrix = pd.DataFrame({b: pts - pts[b] for b in p2_teams})
lead_matrix.columns.name = 'vs'

print("Max points still available:", max_available)
lead_matrix

## Earliest Possible Clinch of P2

In [ ]:
extra_needed = (max_available - lead_matrix + 1).clip(lower=0)
extra_needed = extra_needed.mask(np.eye(len(extra_needed), dtype=bool))

print(extra_needed)
print()
print("Extra lead needed to clinch P2 (against the closest rival):")
print(extra_needed.max(axis=1))

In [ ]:
clinched_p2 = {}
for t in p2_teams:
    best_rival = np.max([cumulative[r] for r in p2_teams if r != t], axis=0)
    clinched_p2[t] = cumulative[t] > best_rival + remaining_max_after

print(remaining_max_after)
print({t: clinched_p2[t].shape for t in p2_teams})